# Wild Drone Appeared — project walkthrough

Fully observable MDP: one policy controls all three drones. Visits deliver inspections of known hazards. This notebook organises the simulator, baseline checks, training and held-out comparison. Run cells in order after installing the project as described in README. No trained-performance claim is implied by the quick runs.

In [ ]:
from pathlib import Path
import subprocess
import sys

repo = Path.cwd()
if repo.name == 'notebooks':
    repo = repo.parent
assert (repo / 'configs/development.json').exists(), 'Open this notebook from the repository'

def run(*arguments):
    subprocess.run([sys.executable, '-m', 'wild_drone', *arguments], cwd=repo, check=True)


## 1. Inspect the state and check the MDP contract

The full fire/fuel/terrain/priority/inspection-age maps and complete drone, charger, wind and mission state are observed. There are 125 joint actions: five commands per drone. Inspection is automatic; staying at base requests charging. See docs/design.md for exact timing and reward components.

In [ ]:
from wild_drone import EnvConfig, WildfireFleetEnv
from gymnasium.utils.env_checker import check_env

env = WildfireFleetEnv(EnvConfig(grid_size=20, horizon=160), render_mode='rgb_array')
check_env(env, skip_render_check=True)
observation, info = env.reset(seed=10000)
{k: v.shape for k, v in observation.items()}


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(13, 8))
plt.imshow(env.render())
plt.axis('off');


## 2. Watch a 40x40 baseline mission

The demo is a baseline, not a trained agent. Orange cells are burning; the right panel shows inspection freshness. Battery levels and charging ownership are displayed. The GIF and metrics are saved under runs/notebook-demo.

In [ ]:
run('demo', '--config', 'configs/main.json', '--policy', 'greedy', '--out', 'runs/notebook-demo')
from IPython.display import Image, display
display(Image(filename=str(repo / 'runs/notebook-demo/demo.gif')))


## 3. Check all three training implementations

This deliberately tiny comparison checks updates, checkpointing and evaluation. It is insufficient training for conclusions. DQN is value-based; REINFORCE uses Monte Carlo policy gradients; A2C uses n-step actor–critic. All receive the same full state and control the whole fleet. The planner has model access and the battery-aware baselines include explicit return heuristics.

In [ ]:
run('compare', '--grid', '8', '--horizon', '12', '--episodes', '2',
    '--training-seeds', '0', '1', '--scenario-seeds', '10000', '10001',
    '--warmup-steps', '4', '--batch-size', '4', '--planner-horizon', '2',
    '--planner-candidates', '3', '--planner-samples', '1', '--out', 'runs/notebook-smoke')
run('plot', '--csv', 'runs/notebook-smoke/comparison.csv', '--out', 'runs/notebook-smoke/comparison.png')
display(Image(filename=str(repo / 'runs/notebook-smoke/comparison.png')))


## 4. Train a development policy

Only run this cell when ready for a longer CPU pilot. Change the algorithm to reinforce or a2c as required. Twenty episodes is a development budget, not a convergence guarantee. Keep validation scenarios separate from the final test set.

In [ ]:
# Uncomment to launch the longer development pilot.
# run('train', '--config', 'configs/development.json', '--algorithm', 'dqn',
#     '--seed', '0', '--out', 'runs/notebook-dqn')
# run('evaluate', '--checkpoint', 'runs/notebook-dqn/policy.pt',
#     '--scenario-seeds', '5000', '5001', '5002', '--out', 'runs/notebook-validation')


## 5. Assignment experiment checklist

1. Review and validate the simulator assumptions and parameter scales.
2. Confirm one algorithm improves over training on the 20x20 development task.
3. Tune all learning methods and planner budgets using only validation scenarios.
4. Freeze settings; train multiple independent seeds and run shared held-out test scenarios.
5. Report freshness, overdue inspections, energy, returns, losses, collisions and decision time.
6. Inspect failure cases and compare reward components; RL may lose to a conventional planner.
7. Add battery/charging/fire sensitivity and reward ablations before publication claims.

Failed missions are padded with zero inspection freshness for the unserved remainder. Final plots aggregate each fitted seed across scenarios first; SD bars are not significance tests.